In [ ]:
# TGraph records carry attributes; convert to geometry only when needed.
def _as_topology(value):
    from topologicpy.Topology import Topology
    from topologicpy.Vertex import Vertex
    if isinstance(value, (list, tuple)):
        return [_as_topology(item) for item in value]
    if not isinstance(value, dict) or "index" not in value or "dictionary" not in value:
        return value
    representation = value.get("representation")
    attributes = value["dictionary"]
    if representation is None and "src" not in value:
        representation = Vertex.ByCoordinates(
            attributes.get("x", 0), attributes.get("y", 0), attributes.get("z", 0))
    if representation is None:
        raise ValueError("This graph record has no geometry representation")
    return Topology.SetDictionary(representation, attributes)


def _record_dictionary(value):
    from topologicpy.Topology import Topology
    if isinstance(value, dict) and "dictionary" in value:
        return value["dictionary"]
    return Topology.Dictionary(value)


def _edge_records(graph):
    from topologicpy.TGraph import TGraph
    records = TGraph.Edges(graph)
    if any(record.get("representation") is None for record in records):
        geometry = TGraph.Edges(graph, asTopologic=True, segmentCurves=False)
        if len(geometry) != len(records):
            raise ValueError("Could not generate one geometry representation per graph edge")
        for record, edge in zip(records, geometry):
            record["representation"] = edge
    return records


In [ ]:
# Import TopologicPy modules. This is not needed on other computers
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

In [ ]:
from topologicpy.Topology import Topology
from topologicpy.CellComplex import CellComplex
from topologicpy.Shell import Shell
from topologicpy.Dictionary import Dictionary
from topologicpy.Color import Color
from topologicpy.TGraph import TGraph

ring = Shell.Pie(radiusA=10, radiusB=2.5, rings=3)
faces = Topology.Faces(_as_topology(ring))

for i, c in enumerate(faces):
    if i < 10:
        value = 10
        c
    elif i < 12:
        value = 20
    elif i < 25:
        value = 30
    elif i < 40:
        value = 40
    elif i < 60:
        value = 50
    elif i < 70:
        value = 60
    else:
        value = 70
    color = Color.ByValueInRange(value, minValue=10, maxValue=70, colorScale="rainbow")
    color = Color.AnyToHex(color)
    d = Dictionary.ByKeysValues(["id", "color"], [value, color])
    c = Topology.SetDictionary(c, d)

Topology.Show(ring, faceOpacity=1, faceColorKey="color")

# Create Quotient Graph
g = TGraph.Quotient(ring,
         topologyType = "Face",
         key = "id",
         weighted = True)

verts = TGraph.Vertices(g)
print("Vertex Dictionaries:")
for v in verts:
    d = _record_dictionary(v)
    value = Dictionary.ValueAtKey(d, "id")
    color = Color.ByValueInRange(value, minValue=10, maxValue=70, colorScale="rainbow")
    color = Color.AnyToHex(color)
    d = Dictionary.SetValueAtKey(d, "color", color)
    d = Dictionary.SetValueAtKey(d, "count", Dictionary.ValueAtKey(d, "count")*3)
    v = Topology.SetDictionary(v, d)
    print(" ",Dictionary.Keys(d), Dictionary.Values(d))

edges = _edge_records(g)
print("Edge Dictionaries:")
for e in edges:
    d = _record_dictionary(e)
    print(" ",Dictionary.Keys(d), Dictionary.Values(d))
Topology.Show(g, vertexSize=9, vertexSizeKey="count", vertexColorKey="color")

flat_g = TGraph.Reshape(g, shape="line 2D")
Topology.Show(flat_g, sagitta=0.3, sides=32, absolute=False, vertexSizeKey="count", vertexColorKey="color", edgeWidth=2, edgeColor="black")
